# Setup (Run Before Attempting Questions) : 

Use the following two fine-tuned sequence classification checkpoints:

DeBERTa: microsoft/deberta-v3-small (fine-tuned checkpoint)

RoBERTa: roberta-base (fine-tuned checkpoint)

Label Mapping
The models output logits for five labels corresponding to the answer options:

Label ID         Option

0                       A

1                       B

2                       C

3                       D

4                       E

# Imports

In [4]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# Device

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


# Model Check points

In [6]:
DEBERTA_CHECKPOINT = "microsoft/deberta-v3-small"
ROBERTA_CHECKPOINT = "roberta-base"

# Load Data

In [7]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print("Train Shape:", train.shape)
print("Test Shape :", test.shape)

train.head()

Train Shape: (2000, 8)
Test Shape : (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [8]:
train["answer"].value_counts().sort_index()

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

# Load Tokenizers

In [9]:
deberta_tokenizer = AutoTokenizer.from_pretrained(DEBERTA_CHECKPOINT)

roberta_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_CHECKPOINT)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Load Models

In [11]:
deberta_model = AutoModelForSequenceClassification.from_pretrained(
    DEBERTA_CHECKPOINT
).to(device)

roberta_model = AutoModelForSequenceClassification.from_pretrained(
    ROBERTA_CHECKPOINT
).to(device)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight     

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
deberta_model.to(device)
deberta_model.eval()
roberta_model.to(device)
roberta_model.eval()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

# Label Mapping

In [15]:
label2id = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}


id2label = {v: k for k, v in label2id.items()}

In [41]:
def get_probabilities(model, tokenizer, text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length = 512
    )
    inputs = {k:v.to(device) for k,v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        
    probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()[0]
    return probs

Load the fine-tuned DeBERTa and RoBERTa models.

For the prompt at row index 25, perform inference using each model independently and apply Softmax to obtain class probabilities.

Question 1:

Which answer option receives the highest probability from the DeBERTa model, and what is that probability?

(answer format : eg - A, probability of A)

In [42]:
row_25 =  train.iloc[25]
# Create input text
text = f"""Question:
{row['prompt']}

A. {row_25['A']}
B. {row_25['B']}
C. {row_25['C']}
D. {row_25['D']}
E. {row_25['E']}
"""
print(text)


# Softmax probabilities
probs = get_probabilities(deberta_model, deberta_tokenizer,text)

# Print probabilities
print("Class Probabilities:")
for i, p in enumerate(probs):
    print(f"{id2label[i]} : {p:.6f}")

# Highest probability
best_idx = probs.argmax()

print("\nHighest Prediction:")
print(f"Answer: {id2label[best_idx]} ,\
probability of {id2label[best_idx]}: {probs[best_idx]:.6f}")

Question:
Choose the correct answer: What is Hesse's principle of transfer in geometry? carefully.

A. Hesse's principle of transfer is a concept in biology that explains the transfer of genetic information from one generation to another.
B. Hesse's principle of transfer is a concept in chemistry that explains the transfer of electrons between atoms in a chemical reaction.
C. Hesse's principle of transfer is a concept in physics that explains the transfer of energy from one object to another.
D. Hesse's principle of transfer is a concept in economics that explains the transfer of wealth from one individual to another.
E. Hesse's principle of transfer is a concept in geometry that states that if the points of the projective line P1 are depicted by a rational normal curve in Pn, then the group of the projective transformations of Pn that preserve the curve is isomorphic to the group of the projective transformations of P1.

Class Probabilities:
A : 0.522461
B : 0.477295

Highest Predicti

Using the same sample (row index 25), average the class probabilities from both models.

Average Probability = [P(DeBERTa) + P(RoBERTa)]/2

Question 2:

Which answer option receives the highest averaged probability after simple probability ensembling?

In [45]:
# Individual model probabilities
deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, text)

# Simple probability averaging
avg_probs = (deberta_probs + roberta_probs) / 2

print("Averaged Probabilities:")
for i, p in enumerate(avg_probs):
    print(i)
    print(f"{id2label[i]} : {p:.6f}")

# Highest averaged probability
best_idx = np.argmax(avg_probs)

print("\nFinal Ensemble Prediction:")
print(f"Answer: {id2label[best_idx]}")
print(f"Average Probability: {avg_probs[best_idx]:.6f}")

Averaged Probabilities:
0
A : 0.516072
1
B : 0.483806

Final Ensemble Prediction:
Answer: A
Average Probability: 0.516072


Apply weighted probability averaging after Softmax using the following weights:

DeBERTa: 0.70

RoBERTa: 0.30

Compute:

P(final) = [0.7 × P(DeBERTa)] + [0.3 × P(RoBERTa)]

Question 3:

Which answer option is ranked first after weighted ensembling?

In [46]:
# Weighted probability averaging
final_probs = (0.70 * deberta_probs) + (0.30 * roberta_probs)

print("Weighted Ensemble Probabilities:")
for i, p in enumerate(final_probs):
    print(f"{id2label[i]} : {p:.6f}")

# Top prediction
best_idx = np.argmax(final_probs)

print("\nWeighted Ensemble Prediction:")
print(f"Answer: {id2label[best_idx]}")
print(f"Probability: {final_probs[best_idx]:.6f}")

Weighted Ensemble Probabilities:
A : 0.518628
B : 0.481324

Weighted Ensemble Prediction:
Answer: A
Probability: 0.518628


Using the weighted ensemble probabilities from Q3, rank all five answer options.

Write the final prediction exactly in Kaggle submission format.

Question 4:

What is the Top-3 prediction string for row index 25?

Example : C A E

In [47]:
# Get indices of probabilities in descending order
top3_indices = np.argsort(final_probs)[::-1][:3]

# Convert indices to option labels
top3_labels = [id2label[idx] for idx in top3_indices]

# Kaggle submission format
top3_prediction = " ".join(top3_labels)

print("Top-3 Prediction:", top3_prediction)

Top-3 Prediction: A B


Run the weighted ensemble pipeline on every row of test.csv.

Save the predictions in a file named submission.csv using the required Kaggle format:

id,prediction

where the prediction column contains the Top-3 ranked options separated by spaces.

Question 5:

Exactly how many prediction rows are present in the generated file (excluding the header)

In [50]:
predictions = []

for _, row in test.iterrows():

    text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    deberta_probs = get_probabilities(deberta_model, deberta_tokenizer, text)
    roberta_probs = get_probabilities(roberta_model, roberta_tokenizer, text)

    final_probs = 0.7 * deberta_probs + 0.3 * roberta_probs

    top3 = np.argsort(final_probs)[::-1][:3]
    prediction = " ".join(id2label[i] for i in top3)

    predictions.append(prediction)

submission = pd.DataFrame({
    "id": test["id"],
    "prediction": predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"\nTotal prediction rows: {len(submission)}")

   id prediction
0   1        A B
1   2        A B
2   3        A B
3   4        A B
4   5        A B

Total prediction rows: 500


For the first 50 rows of test.csv, create two versions of every prompt:

1.Original prompt

2.Instruction-augmented prompt by prepending: "Answer the following multiple-choice question carefully:"

Run inference using DeBERTa on both versions.

Average the predicted probabilities from both passes.

Question 6:

How many of the first 50 rows produce a different Top-1 prediction after applying Test-Time Augmentation?

In [52]:
changed_predictions = 0

for _, row in test.head(50).iterrows():

    # Original prompt
    original_text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # Instruction-augmented prompt
    augmented_text = f"""Answer the following multiple-choice question carefully:

Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # DeBERTa inference
    original_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        original_text
    )

    augmented_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        augmented_text
    )

    # Top-1 before TTA
    original_top1 = np.argmax(original_probs)

    # Average probabilities (TTA)
    tta_probs = (original_probs + augmented_probs) / 2

    # Top-1 after TTA
    tta_top1 = np.argmax(tta_probs)

    # Count changes
    if original_top1 != tta_top1:
        changed_predictions += 1

print(f"Number of changed Top-1 predictions: {changed_predictions}")

Number of changed Top-1 predictions: 1


Process the first 100 rows of test.csv. And compare the Top-1 prediction from:

1. DeBERTa

2. Weighted Ensemble

Question 7:

How many rows have different Top-1 predictions?

In [54]:
different_predictions = 0

for _, row in test.head(100).iterrows():

    text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # DeBERTa probabilities
    deberta_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        text
    )

    # RoBERTa probabilities
    roberta_probs = get_probabilities(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted ensemble (70% DeBERTa + 30% RoBERTa)
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-1 predictions
    deberta_top1 = np.argmax(deberta_probs)
    ensemble_top1 = np.argmax(ensemble_probs)

    # Count differences
    if deberta_top1 != ensemble_top1:
        different_predictions += 1

print(f"Number of different Top-1 predictions: {different_predictions}")

Number of different Top-1 predictions: 0


For the first 100 rows of test.csv, record the highest class probability (confidence) predicted by:

1. DeBERTa

2. Weighted Ensemble

For every row, compute:

Confidence Gain = Ensemble Confidence−DeBERTa Confidence

Question 8:

How many rows have a positive confidence gain (greater than 0)?

In [56]:
positive_confidence_gain = 0

for _, row in test.head(100).iterrows():

    text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # Get probabilities
    deberta_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probabilities(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted ensemble
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Confidence = highest class probability
    deberta_confidence = np.max(deberta_probs)
    ensemble_confidence = np.max(ensemble_probs)

    # Confidence Gain
    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_confidence_gain += 1

print(f"Rows with positive confidence gain: {positive_confidence_gain}")

Rows with positive confidence gain: 3


For the first 100 rows of test.csv, compare the Top-3 prediction strings generated by:

1. DeBERTa alone

2. Weighted Ensemble

Question 9:

How many rows have at least one change in their ordered Top-3 ranking after ensembling?

Examples:

A C D vs. A D C

In [58]:
top3_changes = 0

for _, row in test.head(100).iterrows():

    text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # Get probabilities
    deberta_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probabilities(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted Ensemble
    ensemble_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-3 indices
    deberta_top3 = np.argsort(deberta_probs)[::-1][:3]
    ensemble_top3 = np.argsort(ensemble_probs)[::-1][:3]

    # Convert to labels
    deberta_pred = " ".join(id2label[i] for i in deberta_top3)
    ensemble_pred = " ".join(id2label[i] for i in ensemble_top3)

    # Compare ordered Top-3 strings
    if deberta_pred != ensemble_pred:
        top3_changes += 1

print(f"Rows with changed Top-3 ranking: {top3_changes}")

Rows with changed Top-3 ranking: 0


Using the Top-3 predictions generated by your weighted ensemble for the first 100 validation samples, compute the MAP@3 score.

Question 10:


What is the final MAP@3 score? 

(Round to 4 decimal places.)

In [60]:
# MAP@3 for a single sample
def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1.0 / (predicted.index(actual) + 1)

    return 0.0


scores = []

for _, row in train.head(100).iterrows():

    text = f"""Question:
{row['prompt']}

A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
"""

    # Model probabilities
    deberta_probs = get_probabilities(
        deberta_model,
        deberta_tokenizer,
        text
    )

    roberta_probs = get_probabilities(
        roberta_model,
        roberta_tokenizer,
        text
    )

    # Weighted ensemble
    final_probs = (
        0.70 * deberta_probs +
        0.30 * roberta_probs
    )

    # Top-3 prediction
    top3_idx = np.argsort(final_probs)[::-1][:3]
    top3_pred = [id2label[i] for i in top3_idx]

    # Ground truth
    actual = row["answer"]

    scores.append(apk(actual, top3_pred))

# MAP@3
map3 = np.mean(scores)

print(f"MAP@3: {map3:.4f}")

MAP@3: 0.3300
